# Combiner — Batch Merging & Missing Value Handling

Concatenates the three batch CSVs of Google Review data, isolates records that failed to fetch a rating (incomplete), and routes them back for re-querying.

**Inputs:** `Data/Trial_dataset1_10000_with_GoogleReviews.csv`, `Trial_dataset10001_20000_with_GoogleReviews.csv`, `Trial_dataset22501_25269.csv`  
**Outputs:** `Data/Query_complete.xlsx`, `Data/Quer_file.xlsx`

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

if '__file__' in dir():
    REPO_ROOT = Path(__file__).resolve().parent.parent
else:
    REPO_ROOT = Path().resolve()
os.chdir(REPO_ROOT)
DATA_DIR = REPO_ROOT / "Data"
print(f"Repo root : {REPO_ROOT}")


## 1. Load Batch Files

In [ ]:
REVIEW_COLS = [' CompanyNumber','Google_Overall_Rating','Google_Price_Level',
               'Google_Review_Texts','Google_Individual_Review_Ratings']

df1 = pd.read_csv(DATA_DIR / 'Trial_dataset1_10000_with_GoogleReviews.csv',
                  encoding='latin-1', usecols=REVIEW_COLS)
df2 = pd.read_csv(DATA_DIR / 'Trial_dataset10001_20000_with_GoogleReviews.csv',
                  encoding='latin-1', usecols=REVIEW_COLS)
df3 = pd.read_csv(DATA_DIR / 'Trial_dataset22501_25269.csv', encoding='latin-1')

for name, df in [('Batch 1', df1), ('Batch 2', df2), ('Batch 3', df3)]:
    print(f"{name}: {len(df):,} rows, {df.shape[1]} cols")


## 2. Concatenate & Profile Completeness

In [ ]:
complete_df = pd.concat([df1, df2], axis=0, ignore_index=True)

total      = len(complete_df)
has_rating = complete_df['Google_Overall_Rating'].notna().sum()
missing    = complete_df['Google_Overall_Rating'].isna().sum()

print(f"Combined records  : {total:,}")
print(f"Has Google rating : {has_rating:,}  ({has_rating/total:.1%})")
print(f"Missing rating    : {missing:,}   ({missing/total:.1%})")
print()
print("Null counts per column:")
print(complete_df.isna().sum().to_string())


## 3. Split — Complete vs Incomplete

In [ ]:
incomplete_df = complete_df[complete_df['Google_Overall_Rating'].isna()].copy()
complete_df   = complete_df[complete_df['Google_Overall_Rating'].notna()].copy()

print(f"Complete   (will save as Query_complete.xlsx) : {len(complete_df):,}")
print(f"Incomplete (routed back for re-query)         : {len(incomplete_df):,}")


## 4. Prepare Incomplete Batch for Re-querying

In [ ]:
incomplete_df = incomplete_df.drop(
    columns=['Google_Overall_Rating','Google_Price_Level'], errors='ignore'
)
df3_trimmed = df3.drop(columns=['CompanyName','RegAddress.PostCode'], errors='ignore')
incomplete_df = pd.concat([df3_trimmed, incomplete_df]).drop_duplicates(keep=False)

print(f"Incomplete batch ready for re-query: {len(incomplete_df):,} rows")


## 5. Save Outputs

In [ ]:
complete_df.to_excel(DATA_DIR / 'Query_complete.xlsx', index=False)
incomplete_df.to_excel(DATA_DIR / 'Quer_file.xlsx', index=False)

print(f"✓ Query_complete.xlsx  : {len(complete_df):,} rows")
print(f"✓ Quer_file.xlsx       : {len(incomplete_df):,} rows  (to re-query)")
